In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from preprocessing import preparar_todo_el_dataset

X_train_procesado, X_test_procesado, y_train, y_test, df_limpio, X_train, X_test, pipeline = preparar_todo_el_dataset()

# DEFINIR EL MODELO BASE
# Mantenemos las cosas que sabemos que funcionan bien:
# class_weight='balanced' para proteger a la minoría
# n_jobs=-1 para usar todo el poder de tu PC
rf_base = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=2)

# DEFINIR EL ESPACIO DE BÚSQUEDA 
param_dist = {
    'n_estimators': [50, 100, 150], # Cantidad de árboles en el bosque
    'max_depth': [10, 15, 20,],  # Profundidad máxima
    'min_samples_split': [ 5, 10],      # Clientes mínimos para dividir una rama
    'min_samples_leaf': [ 2, 4]         # Clientes mínimos para crear una hoja final
}

# CONFIGURAR LA BÚSQUEDA ALEATORIA (RandomizedSearchCV)
# n_iter=5: Probará 5 combinaciones al azar de todas las posibles.
# cv=2: Validación cruzada (entrena cada modelo 2 veces en distintas partes de los datos para estar seguro).
# scoring='f1': ¡VITAL! Le decimos que la nota para elegir al ganador sea el F1-Score del "Sí".
random_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=5, 
    cv=2,
    scoring='f1', 
    random_state=42,
    verbose=2 # Nos irá imprimiendo el progreso en pantalla
)



random_search.fit(X_train_procesado, y_train)


print("mejores hiperparámetros")
print(random_search.best_params_)

Fitting 2 folds for each of 5 candidates, totalling 10 fits
[CV] END max_depth=20, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time=   1.5s
[CV] END max_depth=20, min_samples_leaf=4, min_samples_split=10, n_estimators=150; total time=   1.5s
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.9s
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=5, n_estimators=100; total time=   0.9s
[CV] END max_depth=20, min_samples_leaf=2, min_samples_split=5, n_estimators=150; total time=   1.6s
[CV] END max_depth=20, min_samples_leaf=2, min_samples_split=5, n_estimators=150; total time=   1.5s
[CV] END max_depth=20, min_samples_leaf=4, min_samples_split=5, n_estimators=50; total time=   0.5s
[CV] END max_depth=20, min_samples_leaf=4, min_samples_split=5, n_estimators=50; total time=   0.5s
[CV] END max_depth=15, min_samples_leaf=2, min_samples_split=10, n_estimators=100; total time=   0.9s
[CV] END max_depth=15, min_sam

In [3]:
# 1. Extraer el mejor modelo directamente de la búsqueda
mejor_rf = random_search.best_estimator_

# 2. Hacer predicciones con el set de prueba (los datos que el modelo no conoce)
predicciones_mejor_rf = mejor_rf.predict(X_test_procesado)

# 3. Imprimir el veredicto final
print("\n--- MÉTRICAS: RANDOM FOREST OPTIMIZADO ---")
print(classification_report(y_test, predicciones_mejor_rf))

print("\n--- MATRIZ DE CONFUSIÓN ---")
print(confusion_matrix(y_test, predicciones_mejor_rf))


--- MÉTRICAS: RANDOM FOREST OPTIMIZADO ---
              precision    recall  f1-score   support

           0       0.95      0.89      0.92      7251
           1       0.41      0.59      0.48       921

    accuracy                           0.86      8172
   macro avg       0.68      0.74      0.70      8172
weighted avg       0.88      0.86      0.87      8172


--- MATRIZ DE CONFUSIÓN ---
[[6463  788]
 [ 374  547]]


In [4]:
import joblib

# guardar modelo optimizado  (F1-Score 0.48)
joblib.dump(random_search.best_estimator_, 'models/random_forest_optimizado.joblib')

['models/random_forest_optimizado.joblib']